# E-Commerce Data Pipeline - Exploratory Analytics

This notebook demonstrates interactive data exploration using **PySpark** on synthetic e-commerce data generated by `SyntheticDataGenerator`.

### Pipeline Highlights:
- **Customers**: Age normally distributed around 35.
- **Orders**: Pareto distributed (20% of customers generate ~80% of orders).
- **Products**: Categories (`Electronics`, `Clothing`, `Home`, `Sports`, `Books`), price range $10–$500, rating 1–5.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import config
from src.spark_analytics import (
    get_spark_session,
    load_raw_datasets,
    calculate_revenue_by_category,
    calculate_top_customers,
    calculate_monthly_sales_trend,
    calculate_customer_age_demographics,
    calculate_top_products
)

# Initialize PySpark Session
spark = get_spark_session()
spark

## 1. Load Raw Datasets
Ingest CSV files into PySpark DataFrames with strict schemas.

In [ ]:
dfs = load_raw_datasets(spark)
for name, df in dfs.items():
    print(f"Dataset: {name:10s} | Rows: {df.count():,} | Columns: {len(df.columns)}")
    df.printSchema()
    print("-" * 60)

## 2. Business Insight: Revenue by Product Category

In [ ]:
category_rev_df = calculate_revenue_by_category(dfs["orders"], dfs["products"])
category_rev_df.show(truncate=False)

category_pandas = category_rev_df.toPandas()
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.barplot(data=category_pandas, x="category", y="total_revenue", palette="viridis")
plt.title("Total Realized Revenue by Product Category", fontsize=14)
plt.xlabel("Category", fontsize=12)
plt.ylabel("Revenue ($)", fontsize=12)
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 3. Business Insight: Top 10 Spending Customers (Pareto 80/20 Analysis)

In [ ]:
top_customers_df = calculate_top_customers(
    dfs["customers"], dfs["orders"], dfs["products"], top_n=10
)
top_customers_df.show(truncate=False)

## 4. Business Insight: Customer Age Demographics

In [ ]:
demographics_df = calculate_customer_age_demographics(
    dfs["customers"], dfs["orders"], dfs["products"]
)
demographics_df.show(truncate=False)

demo_pandas = demographics_df.toPandas()
plt.figure(figsize=(9, 4))
sns.barplot(data=demo_pandas, x="age_group", y="total_revenue", palette="mako")
plt.title("Total Revenue by Customer Age Bracket", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Stop PySpark Session

In [ ]:
spark.stop()
print("SparkSession stopped successfully.")